# 🤖 ETH/USD RL Trading Agent — PPO + Regime Detection

**Fixes vs original:**
- `VecNormalize` wraps train env → obs on same scale → MLP can actually learn
- `ent_coef=0.05` (was 0.01) → entropy lives long enough for exploration
- Step-wise reward = unrealised PnL per candle → dense gradient signal
- Confidence gate lowered to 0.35 default (or disable for greedy eval)
- Annualised Sharpe uses `sqrt(252 × 24)` (hourly data)

## 1 · Install

In [1]:
!pip install ccxt pandas stable-baselines3 gymnasium ta scipy -q

  DEPRECATION: Building 'ta' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'ta'. Discussion can be found at https://github.com/pypa/pip/issues/6334


## 2 · Fetch ETH/USD OHLCV (1 h, 2025)

In [ ]:
import ccxt
import pandas as pd
import numpy as np
import time

exchange = ccxt.coinbase()

all_ohlcv = []
since = exchange.parse8601('2025-01-01T00:00:00Z')
end   = exchange.parse8601('2025-12-31T23:59:59Z')

print('Fetching ETH/USD 1h from Coinbase...')
while True:
    batch = exchange.fetch_ohlcv('ETH/USD', '1h', since=since, limit=500)
    if not batch:
        break
    all_ohlcv += batch
    since = batch[-1][0] + 1
    print(f'  {len(all_ohlcv):,} candles...', end='\r')
    time.sleep(0.4)
    if since > end:
        break

df = pd.DataFrame(all_ohlcv,
                  columns=['timestamp','open','high','low','close','volume'])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
df.set_index('timestamp', inplace=True)
df = df.astype(float)
df = df[~df.index.duplicated()]

print(f'\nRows: {len(df):,}  |  {df.index[0]} → {df.index[-1]}')
df.head()

## 3 · Feature engineering

In [ ]:
import ta

# --- Price action features ---
df['returns']         = df['close'].pct_change()
df['ma10']            = df['close'].rolling(10).mean()
df['ma30']            = df['close'].rolling(30).mean()
df['ma_ratio']        = df['ma10'] / df['ma30']           # >1 = bullish MA cross
df['volatility']      = df['returns'].rolling(20).std()
df['rsi']             = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
df['norm_price']      = (df['close'] - df['close'].min()) / (df['close'].max() - df['close'].min())

# --- Regime features ---
df['price_momentum']  = df['close'].pct_change(20)
df['ma_slope']        = df['ma10'].diff(5) / df['ma10'].shift(5)
df['trend_dir']       = (df['ma10'] - df['ma30']) / df['ma30']

df.dropna(inplace=True)
print(f'Features ready: {df.shape}')
print(df[['close','returns','rsi','volatility','ma_ratio']].describe().round(4))

## 4 · Market regime detection

In [ ]:
from scipy import stats

vol_high = df['volatility'].quantile(0.66)

def classify(row):
    if row['volatility'] > vol_high:
        return 'volatile'
    elif abs(row['price_momentum']) > 0.03 or abs(row['ma_slope']) > 0.005:
        return 'trending'
    return 'ranging'

df['regime_label'] = df.apply(classify, axis=1)
print('Raw:\n', df['regime_label'].value_counts())

df['regime_code'] = df['regime_label'].map({'ranging': 0, 'trending': 1, 'volatile': 2})
df['regime_code'] = (
    df['regime_code']
    .rolling(168, min_periods=1)
    .apply(lambda x: stats.mode(x, keepdims=True)[0][0])
    .astype(int)
)
df['regime_label'] = df['regime_code'].map({0: 'ranging', 1: 'trending', 2: 'volatile'})
print('Smoothed:\n', df['regime_label'].value_counts())

### 4a · Visualise regimes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COLORS = {'ranging': '#4444FF', 'trending': '#00AA00', 'volatile': '#FF4444'}
fig, ax = plt.subplots(figsize=(16, 5))
plot_df = df.reset_index(drop=True)

ax.plot(plot_df['close'].values, color='white', linewidth=0.8, zorder=2)

regime_arr = plot_df['regime_label'].values
i = 0
while i < len(regime_arr):
    regime = regime_arr[i]
    j = i
    while j < len(regime_arr) and regime_arr[j] == regime:
        j += 1
    ax.axvspan(i, j, alpha=0.4, color=COLORS.get(regime, 'grey'), zorder=0)
    i = j

patches = [mpatches.Patch(color=v, label=k, alpha=0.7) for k, v in COLORS.items()]
ax.legend(handles=patches, loc='upper right', fontsize=11)
ax.set_facecolor('#111111')
fig.patch.set_facecolor('#111111')
ax.set_title('ETH/USD — Market Regime Overlay (2025)', color='white', fontsize=14)
ax.set_xlabel('Candle index (1 h)', color='grey')
ax.set_ylabel('Price (USD)', color='grey')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15, color='white')
plt.tight_layout()
plt.show()

## 5 · Trading environment

**Key design decisions:**
- **Dense reward** = step-wise unrealised P&L (not just on sell). Sparse rewards → dead gradients.
- **Position flag** in obs so agent knows if it's holding.
- **`VecNormalize`** applied at train time to equalise feature scales (RSI 0–100 vs returns ±0.01).

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

OBS_COLS = ['returns', 'ma_ratio', 'volatility', 'rsi', 'norm_price', 'trend_dir']
TC       = 0.0005   # 0.05% transaction cost per trade


class TradingEnv(gym.Env):
    """
    Long-only ETH/USD hourly trading env.

    Actions: 0=Hold  1=Buy  2=Sell

    Reward:
      - Every step while in position: step-wise unrealised P&L  ← DENSE
      - On Sell: realised P&L (already captured step-wise, no double-count)
      - Transaction cost on Buy/Sell
      - Auto stop-loss at -3%: force-exit, no extra penalty (P&L covers it)
    """
    metadata = {'render_modes': []}

    def __init__(self, df: pd.DataFrame):
        super().__init__()
        self.df = df[OBS_COLS + ['close']].reset_index(drop=True)
        self.n  = len(self.df)

        n_obs = len(OBS_COLS) + 1   # features + position flag
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(n_obs,), dtype=np.float32)
        self.action_space = spaces.Discrete(3)
        self._reset_state()

    # ── internals ─────────────────────────────────────────────────────
    def _reset_state(self):
        self.current_step = 0
        self.position     = 0
        self.entry_price  = 0.0
        self.prev_price   = self.df['close'].iloc[0]
        self.total_profit = 0.0

    def _get_obs(self) -> np.ndarray:
        row = self.df[OBS_COLS].iloc[self.current_step].values
        obs = np.append(row, float(self.position)).astype(np.float32)
        return np.nan_to_num(obs)

    # ── gym API ───────────────────────────────────────────────────────
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_state()
        return self._get_obs(), {}

    def step(self, action: int):
        price = self.df['close'].iloc[self.current_step]

        reward = 0.0

        if action == 1 and self.position == 0:      # ── BUY
            self.position    = 1
            self.entry_price = price
            reward          -= TC

        elif action == 2 and self.position == 1:    # ── SELL
            # Realised P&L not re-added (already collected step-wise)
            reward            = -TC
            self.total_profit += (price - self.entry_price) / (self.entry_price + 1e-9)
            self.position     = 0

        # ── DENSE: step-wise unrealised P&L while holding ────────────
        if self.position == 1:
            step_ret = (price - self.prev_price) / (self.prev_price + 1e-9)
            reward  += step_ret

            # Auto stop-loss at -3% from entry
            unreal = (price - self.entry_price) / (self.entry_price + 1e-9)
            if unreal < -0.03:
                self.total_profit += unreal
                reward             = unreal - TC
                self.position      = 0

        self.prev_price    = price
        self.current_step += 1
        done = self.current_step >= self.n - 1
        return self._get_obs(), float(reward), done, False, {}

    def render(self): pass


# ── Sanity check ──────────────────────────────────────────────────────
_e = TradingEnv(df)
obs, _ = _e.reset()
print('obs shape :', obs.shape)
print('obs values:', np.round(obs, 4))
print('action    :', _e.action_space)

## 6 · Train PPO with VecNormalize

`VecNormalize` keeps running mean/std of observations and normalises them to ~N(0,1).  
This is the **single most important fix** — without it, RSI dominates and the MLP can't learn from tiny return values.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

SPLIT    = int(len(df) * 0.8)
train_df = df.iloc[:SPLIT].copy()
test_df  = df.iloc[SPLIT:].reset_index(drop=True)

print(f'Train: {len(train_df):,} candles  ({df.index[0]} → {df.index[SPLIT]})')
print(f'Test:  {len(test_df):,}  candles  ({df.index[SPLIT]} → {df.index[-1]})')

# Wrap in VecNormalize — normalises obs to ~N(0,1) using running stats
raw_env   = DummyVecEnv([lambda: TradingEnv(train_df)])
train_env = VecNormalize(raw_env, norm_obs=True, norm_reward=False, clip_obs=5.0)

model = PPO(
    'MlpPolicy', train_env,
    learning_rate  = 3e-4,
    n_steps        = 2048,
    batch_size     = 64,
    n_epochs       = 10,
    gamma          = 0.99,
    ent_coef       = 0.05,   # was 0.01 — keeps entropy alive during training
    clip_range     = 0.2,
    verbose        = 1,
    policy_kwargs  = dict(net_arch=[256, 256]),  # bigger head for 7-dim obs
)

model.learn(total_timesteps=300_000)

# Save both model AND normalisation stats (needed for correct test inference)
model.save('ppo_eth_trading')
train_env.save('vecnorm_stats.pkl')
print('\nSaved → ppo_eth_trading.zip  +  vecnorm_stats.pkl')

### 6a · Diagnose policy — action probs on test obs

In [ ]:
import torch
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Build normalised test env using TRAIN stats (do not update!)
raw_test   = DummyVecEnv([lambda: TradingEnv(test_df)])
test_env_n = VecNormalize.load('vecnorm_stats.pkl', raw_test)
test_env_n.training  = False   # freeze running stats
test_env_n.norm_reward = False

obs = test_env_n.reset()
all_probs = []
for _ in range(300):
    obs_t = torch.tensor(obs, dtype=torch.float32)
    with torch.no_grad():
        dist  = model.policy.get_distribution(obs_t)
        probs = dist.distribution.probs.numpy()[0]
    all_probs.append(probs)
    action, _ = model.predict(obs, deterministic=False)
    obs, _, done, _ = test_env_n.step(action)
    if done[0]:
        obs = test_env_n.reset()

import numpy as np
ap = np.array(all_probs)
print('Mean action probs over 300 steps:')
print(f'  Hold (0): {ap[:,0].mean():.3f}')
print(f'  Buy  (1): {ap[:,1].mean():.3f}')
print(f'  Sell (2): {ap[:,2].mean():.3f}')
print()
print('If Buy+Sell mean < 0.10 → entropy collapsed, re-run training.')
print('If Buy+Sell mean > 0.15 → policy is alive, proceed to eval.')

## 7 · Greedy evaluation (no confidence gate)

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Re-use normalised test env from cell above
obs       = test_env_n.reset()
done_flag = False
rewards, actions_log = [], []

while not done_flag:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _ = test_env_n.step(action)
    rewards.append(float(reward[0]))
    actions_log.append(int(action[0]))
    done_flag = bool(done[0])

cum_rewards = np.cumsum(rewards)
bh_returns  = test_df['returns'].cumsum().values[:len(rewards)]
counts      = Counter(actions_log)

# ── Plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 11), facecolor='#111111')

axes[0].plot(cum_rewards, label='RL Agent',   color='#00AAFF', linewidth=1.5)
axes[0].plot(bh_returns,  label='Buy & Hold', color='#FF6600', linestyle='--', linewidth=1.2)
axes[0].set_title('Cumulative Returns', color='white', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_facecolor('#1a1a1a'); axes[0].tick_params(colors='white')
axes[0].grid(True, alpha=0.2)

axes[1].plot(test_df['close'].values[:len(rewards)], color='#AAAAAA', linewidth=0.8)
buy_i  = [i for i, a in enumerate(actions_log) if a == 1]
sell_i = [i for i, a in enumerate(actions_log) if a == 2]
axes[1].scatter(buy_i,  test_df['close'].iloc[buy_i],  marker='^', color='#00FF88', s=20, label='Buy')
axes[1].scatter(sell_i, test_df['close'].iloc[sell_i], marker='v', color='#FF4444', s=20, label='Sell')
axes[1].set_title('ETH Price + Agent Signals', color='white', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].set_facecolor('#1a1a1a'); axes[1].tick_params(colors='white')
axes[1].grid(True, alpha=0.2)

bars = axes[2].bar(['Hold', 'Buy', 'Sell'],
                   [counts[0], counts[1], counts[2]],
                   color=['#556677', '#00AA66', '#CC3333'], edgecolor='white')
for b in bars:
    axes[2].text(b.get_x() + b.get_width()/2,
                 b.get_height() + 1, f'{int(b.get_height()):,}',
                 ha='center', color='white', fontsize=10)
axes[2].set_title('Action Distribution', color='white', fontsize=12)
axes[2].set_facecolor('#1a1a1a'); axes[2].tick_params(colors='white')
axes[2].grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

print(f'RL P&L      : {cum_rewards[-1]:+.4f}')
print(f'B&H P&L     : {bh_returns[-1]:+.4f}')
print(f'Trades      : {counts[1]+counts[2]}  (Buy={counts[1]}  Sell={counts[2]})')

## 8 · Confidence-gated evaluation

Gate threshold of **0.35** (not 0.60 — PPO Categorical probs are smoother than Gaussian).  
If still 0 trades → raise threshold to 0.99 and check if *any* Buy/Sell appears.

In [ ]:
CONFIDENCE_THRESHOLD = 0.35   # lower = more trades; raise to debug

def predict_gated(model, obs_vec, threshold):
    obs_t = torch.tensor(obs_vec, dtype=torch.float32)
    with torch.no_grad():
        dist  = model.policy.get_distribution(obs_t)
        probs = dist.distribution.probs.numpy()[0]
    best_prob = float(probs.max())
    action    = int(probs.argmax())
    if best_prob < threshold:
        return 0, best_prob
    return action, best_prob


obs        = test_env_n.reset()
rewards2, actions2, confs = [], [], []

done_flag = False
while not done_flag:
    action, conf = predict_gated(model, obs, CONFIDENCE_THRESHOLD)
    obs, reward, done, _ = test_env_n.step(np.array([action]))
    rewards2.append(float(reward[0]))
    actions2.append(action)
    confs.append(conf)
    done_flag = bool(done[0])

c2   = Counter(actions2)
bh2  = test_df['returns'].cumsum().values[len(rewards2) - 1]

print(f'Threshold    : {CONFIDENCE_THRESHOLD:.0%}')
print(f'RL P&L (gate): {sum(rewards2):+.4f}')
print(f'B&H P&L      : {bh2:+.4f}')
print(f'Avg conf     : {np.mean(confs):.3f}')
print(f'Trades       : Buy={c2[1]}  Sell={c2[2]}')
print(f'Held         : {c2[0]}')

## 9 · Performance metrics

In [ ]:
def compute_metrics(rewards_arr, label='Strategy'):
    r   = np.array(rewards_arr)
    cum = np.cumsum(r)
    sharpe   = (r.mean() / (r.std() + 1e-9)) * np.sqrt(252 * 24)  # hourly annualised
    peak     = np.maximum.accumulate(cum)
    max_dd   = (cum - peak).min()
    win_rate = (r > 0).mean()
    print(f'\n══ {label} ══')
    print(f'  Total P&L   : {cum[-1]:+.4f}')
    print(f'  Sharpe (ann): {sharpe:.3f}')
    print(f'  Max Drawdown: {max_dd:.4f}')
    print(f'  Win Rate    : {win_rate*100:.1f}%')
    return dict(pnl=float(cum[-1]), sharpe=sharpe, max_dd=max_dd, win_rate=win_rate)

m1 = compute_metrics(rewards,  'RL Greedy')
m2 = compute_metrics(rewards2, 'RL Confidence-Gated')
m3 = compute_metrics(test_df['returns'].values[:len(rewards)], 'Buy & Hold')